# Generality — how errors and corrections overlap

Reliability already measured whether ReasonEdit fixes the sample that was edited. Generality asks a different question: if we edit some samples, does that correction carry to other samples that failed the same way.

This notebook only looks at the relationship between model errors and human corrections. It uses the same six edit sets as the reliability runs (3 dermatologists × MedGemma and DermatoLlama): a row is a completed top-1 error whose y16 class differs from the human correction.

The question is the same on every row. `model_answer` / `human_answer` are the raw model top-1 and the canonical human correction ReasonEdit would insert. `*_y16` is the coarser 16-class label.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

SHARED = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/shared")
sys.path.insert(0, str(SHARED))
import config as C

rows = []
for path in sorted(C.OUTPUT_DIR.glob("edit_set__*.jsonl")):
    for line in path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))

df = pd.DataFrame(rows).rename(columns={
    "case_id": "sample_id",
    "login_id": "user",
    "model_top1": "model_answer",
    "model_top1_y16": "model_y16",
    "human_top1_raw": "human_answer_raw",
    "answer": "human_answer",
    "human_top1_y16": "human_y16",
})
keep = [
    "sample_id", "user", "model", "question",
    "model_answer", "model_y16", "human_answer_raw", "human_answer", "human_y16",
]
edits = df[keep].sort_values(["model", "user", "sample_id"]).reset_index(drop=True)

print(f"{len(edits)} edits, {edits['sample_id'].nunique()} samples, {edits['question'].nunique()} distinct question")
print(edits.groupby(["model", "user"]).size().unstack(fill_value=0))

## All edits

One row per edit. The question column is included; it does not vary.

In [ ]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
edits

## Distributions

Per model: what the model said when it was wrong, what humans changed it to, and the transition between the two. Counts are edits, so a sample reviewed by two users is counted twice.

In [ ]:
def counts(frame, column):
    return frame[column].value_counts().rename("n").to_frame()

for model, g in edits.groupby("model"):
    print(f"\n===== {model}  ({len(g)} edits, {g['sample_id'].nunique()} samples) =====")
    print("\nmodel answer (y16)")
    print(counts(g, "model_y16"))
    print("\nhuman answer (y16)")
    print(counts(g, "human_y16"))
    print("\nhuman answer (canonical text ReasonEdit inserts)")
    print(counts(g, "human_answer"))
    transition = (
        g.groupby(["model_y16", "human_y16"]).size()
        .rename("n").reset_index()
        .sort_values("n", ascending=False)
    )
    print("\ntransition  model_y16 -> human_y16")
    print(transition.to_string(index=False))

## Overlaps

Three grains, still split by model:

1. **Same sample.** More than one user edited this case.
2. **Model fails the same way.** Same `model_y16`, regardless of the correction. `n_human_y16` is how many different corrections sit inside that failure.
3. **Same failure and same correction.** Same `(model_y16, human_y16)`, then the stricter `(model_answer, human_answer)`. A cluster with more than one sample is a candidate generality group: edit some of them, hold out the rest.

In [ ]:
def cluster(frame, keys):
    out = (
        frame.groupby(keys)
        .agg(
            n_edits=("sample_id", "size"),
            n_samples=("sample_id", "nunique"),
            n_users=("user", "nunique"),
            samples=("sample_id", lambda s: ", ".join(sorted(set(s)))),
            users=("user", lambda s: ", ".join(sorted(set(s)))),
        )
        .reset_index()
        .sort_values(["n_edits", "n_samples"], ascending=False)
    )
    return out

for model, g in edits.groupby("model"):
    print(f"\n===== {model} =====")

    per_sample = g.groupby("sample_id")["user"].nunique()
    shared_ids = per_sample[per_sample > 1].index
    shared = g[g["sample_id"].isin(shared_ids)].sort_values("sample_id")
    print(f"\nsame sample, multiple users: {len(shared_ids)}")
    if len(shared):
        print(shared[["sample_id", "user", "model_answer", "human_answer_raw", "human_answer", "human_y16"]].to_string(index=False))

    fail = cluster(g, ["model_y16"])
    n_corr = g.groupby("model_y16")["human_y16"].nunique().rename("n_human_y16")
    fail = fail.merge(n_corr, on="model_y16")
    print("\nmodel fails the same way (model_y16)")
    print(fail.to_string(index=False))

    same_y16 = cluster(g, ["model_y16", "human_y16"])
    print("\nsame failure and same y16 correction, n_samples >= 2")
    print(same_y16[same_y16["n_samples"] >= 2].to_string(index=False))

    same_text = cluster(g, ["model_answer", "human_answer"])
    print("\nsame failure and same canonical correction, n_samples >= 2")
    print(same_text[same_text["n_samples"] >= 2].to_string(index=False))